# 糖尿病与胆结石
在 Colab 选择 **Runtime → Run all**。NHANES 使用 CDC 2017–2020 年 3 月公开数据。MR 从内置的 109 个已对齐 SNP 开始，暴露为 GCST006867，结局为 FinnGen finn-b-K11_CHOLELITH。此前的 GWAS 筛选、LD clumping 和等位基因对齐在原始笔记本中完成。

In [ ]:
from pathlib import Path
from urllib.request import urlopen

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
folder = str(DATA_DIR) + "/"
base_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles"
for name in ("DEMO", "MCQ", "DIQ", "GHB", "GLU", "SMQ", "ALQ", "BMX"):
    filename = f"P_{name}.xpt"
    path = DATA_DIR / filename
    if not path.exists():
        with urlopen(f"{base_url}/{filename}", timeout=90) as response:
            data_bytes = response.read()
        if not data_bytes.startswith(b"HEADER RECORD"):
            raise ValueError(f"CDC did not return an XPT file for {filename}")
        path.write_bytes(data_bytes)
print("NHANES files ready")


# Download the P_DEMO file


# store the dataset and print its size

In [ ]:
import pandas as pd

demo = pd.read_sas(folder +"P_DEMO.xpt", format="xport")

print(demo.shape)
demo.head()

# select the variables that are going to be used

In [ ]:
demo[
    [
        "SEQN",
        "RIAGENDR",
        "RIDAGEYR",
        "RIDRETH1",
        "RIDRETH3",
        "INDFMPIR",
        "SDMVPSU",
        "SDMVSTRA",
        "WTINTPRP",
        "WTMECPRP"
    ]
].head(10)

# Add the selected variables to a new dataframe


In [ ]:
demo_keep = demo[
    [
        "SEQN",
        "RIAGENDR",
        "RIDAGEYR",
        "RIDRETH1",
        "RIDRETH3",
        "INDFMPIR",
        "SDMVPSU",
        "SDMVSTRA",
        "WTINTPRP",
        "WTMECPRP"
    ]
].copy()
demo_keep.head()

# read the file about medical conditions(P_MCQ)

In [ ]:
mcq = pd.read_sas(folder +"P_MCQ.xpt", format="xport")

print(mcq.shape)
mcq[["SEQN", "MCQ550"]].head(10)

# gallstone variable analysis

In [ ]:
mcq["MCQ550"].value_counts(dropna=False)

In [ ]:
mcq_keep = mcq[["SEQN", "MCQ550"]].copy()

In [ ]:
import numpy as np

mcq_keep["gallstone"] = mcq_keep["MCQ550"].map({
    1: 1,
    2: 0
})

In [ ]:
mcq_keep["gallstone"].value_counts(dropna=False)

#Merge the P_demo file and the P_MCQ file together accorfing to their SEQN.

In [ ]:
data = demo_keep.merge(
    mcq_keep[["SEQN", "gallstone"]],
    on="SEQN",
    how="left"
)

data.shape

In [ ]:
data[["SEQN", "RIDAGEYR", "RIAGENDR", "gallstone"]].head(10)

In [ ]:
data["gallstone"].value_counts(dropna=False)

#Download the file about diabetes status（P_DIQ）

In [ ]:
diq = pd.read_sas(folder +"P_DIQ.xpt", format="xport")
diq.shape

In [ ]:
diq[["SEQN", "DIQ010"]].head(10)


In [ ]:
diq["DIQ010"].value_counts(dropna=False)

In [ ]:
print(diq.shape)
print(diq.columns.tolist())
print(diq["DIQ010"].value_counts(dropna=False))

Create a variable that records whether each patient has diabetes

In [ ]:
import numpy as np

diq_keep = diq[['SEQN', 'DIQ010']].copy()

diq_keep['diabetes'] = diq_keep['DIQ010'].map({
    1: 1,
    2: 0
})

print(diq_keep['diabetes'].value_counts(dropna=False))

#Merge the DIQ file with the other files

In [ ]:
df = demo_keep.merge(
    mcq_keep[['SEQN', 'gallstone']],
    on='SEQN',
    how='left'
)

df = df.merge(
    diq_keep[['SEQN', 'diabetes']],
    on='SEQN',
    how='left'
)

print(df.shape)
print(df[['SEQN', 'diabetes', 'gallstone']].head())

#Relationship between the number of diabetes and gallstone patients

In [ ]:
import pandas as pd

print(pd.crosstab(
    df['diabetes'],
    df['gallstone'],
    margins=True
))

#unweighted gallstone prevalence by diabetes status

In [ ]:
prevalence = (
    df.groupby('diabetes')['gallstone']
    .mean() * 100
)

print("Gallstone prevalence by diabetes status:")
print(prevalence)

# Calculate unadjusted odds ratio

In [ ]:
# Diabetes group
odds_diabetes = 259 / 1152

# Non-diabetes group
odds_no_diabetes = 709 / 6824

# Odds ratio
OR = odds_diabetes / odds_no_diabetes

print(f"Unadjusted Odds Ratio: {OR:.3f}")

#95% Confidence Interval

In [ ]:
import numpy as np

# Create contingency table
table = pd.crosstab(df['diabetes'], df['gallstone'])

# Extract values
a = table.loc[1, 1]  # Diabetes + Gallstones
b = table.loc[1, 0]  # Diabetes + No gallstones
c = table.loc[0, 1]  # No diabetes + Gallstones
d = table.loc[0, 0]  # No diabetes + No gallstones

# Calculate odds ratio
OR = (a * d) / (b * c)

# Calculate standard error of log(OR)
SE = np.sqrt(1/a + 1/b + 1/c + 1/d)

# Calculate 95% confidence interval
lower = np.exp(np.log(OR) - 1.96 * SE)
upper = np.exp(np.log(OR) + 1.96 * SE)

print(f"OR: {OR:.3f}")
print(f"95% CI: [{lower:.3f}, {upper:.3f}]")

### Interpretation of Unadjusted Odds Ratio

Participants with diabetes had significantly higher odds of
gallstones compared with participants without diabetes
(OR = 2.164, 95% CI: 1.852–2.528).

However, this analysis was unweighted and did not adjust for
potential confounding variables, such as age, sex, and BMI.

#Import P_GHB file (Glycohemoglobin)

In [ ]:
import pandas as pd

ghb = pd.read_sas(folder +"P_GHB.xpt", format="xport")

print(ghb.shape)
print(ghb.columns.tolist())

In [ ]:
# Summary statistics for HbA1c
print(ghb['LBXGH'].describe())

# Check missing values
print("Missing values:", ghb['LBXGH'].isna().sum())

# Count participants with HbA1c >= 6.5%
print("HbA1c >= 6.5%:", (ghb['LBXGH'] >= 6.5).sum())

#Merge the file

In [ ]:
# Merge HbA1c into the existing dataset
df = df.merge(
    ghb[['SEQN', 'LBXGH']],
    on='SEQN',
    how='left'
)

print(df.shape)
print(df[['SEQN', 'diabetes', 'gallstone', 'LBXGH']].head())

# Compare the diagnosis based on high glycohemoglobin value and self-reported diabtes

In [ ]:
# Classify HbA1c results
df['hba1c_high'] = (df['LBXGH'] >= 6.5).astype('Int64')
df.loc[df['LBXGH'].isna(), 'hba1c_high'] = pd.NA

# Compare self-reported diabetes and HbA1c
print(pd.crosstab(
    df['diabetes'],
    df['hba1c_high'],
    margins=True
))

# Categorize the patient as diabetes if he has glycohemoglobin>=6.5 or has self-reported diabetes

In [ ]:
import numpy as np
import pandas as pd

# Create a new diabetes variable
df['diabetes_new'] = np.nan

# Diabetes if self-reported or HbA1c >= 6.5%
df.loc[
    (df['diabetes'] == 1) | (df['LBXGH'] >= 6.5),
    'diabetes_new'
] = 1

# No diabetes if self-report is negative AND HbA1c < 6.5%
df.loc[
    (df['diabetes'] == 0) & (df['LBXGH'] < 6.5),
    'diabetes_new'
] = 0

print(df['diabetes_new'].value_counts(dropna=False))

# Import the P_GLU file(Fasting Plasma Glucose)

In [ ]:
import pandas as pd

# Read fasting glucose data
glu = pd.read_sas(folder +"P_GLU.xpt", format="xport")

print(glu.shape)
print(glu.columns.tolist())

In [ ]:
glu_keep = glu[['SEQN', 'LBXGLU', 'WTSAFPRP']].copy()

In [ ]:
# Summary statistics
print(glu['LBXGLU'].describe())

# Missing values
print("Missing values:", glu['LBXGLU'].isna().sum())

# Participants meeting diabetes threshold
print("Fasting glucose >= 126 mg/dL:",
      (glu['LBXGLU'] >= 126).sum())

#Merge the file

In [ ]:
# Merge fasting glucose data
df = df.merge(
    glu[['SEQN', 'LBXGLU', 'WTSAFPRP']],
    on='SEQN',
    how='left',
    validate='one_to_one'
)

print(df.shape)
print(df[['SEQN', 'diabetes', 'LBXGH', 'LBXGLU']].head())

#任何一个条件满足就归为diabetes 此条件为空腹血糖>=126

In [ ]:
import numpy as np

# Preserve the original self-reported diabetes variable
# Create a new variable based on all three indicators

df['diabetes_combined'] = np.nan

positive = (
    (df['diabetes'] == 1) |
    (df['LBXGH'] >= 6.5) |
    (df['LBXGLU'] >= 126)
)

negative = (
    (df['diabetes'] == 0) &
    (df['LBXGH'] < 6.5) &
    (
        (df['LBXGLU'] < 126) |
        (df['LBXGLU'].isna())
    )
)

df.loc[positive, 'diabetes_combined'] = 1
df.loc[negative, 'diabetes_combined'] = 0

print(df['diabetes_combined'].value_counts(dropna=False))

In [ ]:
print(pd.crosstab(
    df['diabetes'],
    df['diabetes_combined'],
    margins=True
))

# Calculate new unadjusted OR ratio using Combined Diabetes Definition

In [ ]:
# Compare combined diabetes status with gallstone status

table_new = pd.crosstab(
    df['diabetes_combined'],
    df['gallstone'],
    margins=True
)

print(table_new)

In [ ]:
import numpy as np
import pandas as pd

# Calculate gallstone prevalence
prevalence_new = (
    df.groupby('diabetes_combined')['gallstone']
    .mean() * 100
)

print("Gallstone prevalence (%):")
print(prevalence_new)

# Create contingency table
table_new = pd.crosstab(
    df['diabetes_combined'],
    df['gallstone']
)

a = table_new.loc[1, 1]
b = table_new.loc[1, 0]
c = table_new.loc[0, 1]
d = table_new.loc[0, 0]

# Calculate odds ratio
OR_new = (a * d) / (b * c)

# Calculate standard error
SE_new = np.sqrt(1/a + 1/b + 1/c + 1/d)

# Calculate 95% confidence interval
lower_new = np.exp(np.log(OR_new) - 1.96 * SE_new)
upper_new = np.exp(np.log(OR_new) + 1.96 * SE_new)

print(f"OR: {OR_new:.3f}")
print(f"95% CI: [{lower_new:.3f}, {upper_new:.3f}]")

# Create a logistic regression model to investigate the relashionship between diabetes and gallstone using NHANES dataset

In [ ]:
# Check demographic variables
columns = [
    'RIDAGEYR',  # Age
    'RIAGENDR',  # Sex
    'RIDRETH3'   # Race/ethnicity
]

print(df[columns].describe())
print(df[columns].isna().sum())

In [ ]:
# Correct near-zero age values
df.loc[df['RIDAGEYR'] < 1, 'RIDAGEYR'] = 0

print(df['RIDAGEYR'].min())
print((df['RIDAGEYR'] == 0).sum())

In [ ]:
# Select adults aged 20 and above
df_adult = df[df['RIDAGEYR'] >= 20].copy()

print("Total participants:", len(df))
print("Adult participants:", len(df_adult))

print("Adults with valid diabetes and gallstone data:",
      df_adult[['diabetes_combined', 'gallstone']].dropna().shape[0])

# Logistic Regression Adjusted for Age, Sex, and Ethnicity

In [ ]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Select variables and remove missing values
model_data = df_adult[
    ['gallstone', 'diabetes_combined',
     'RIDAGEYR', 'RIAGENDR', 'RIDRETH3']
].dropna().copy()

print("Sample size:", len(model_data))

# Fit logistic regression
model1 = smf.logit(
    formula="""
    gallstone ~ diabetes_combined
    + RIDAGEYR
    + C(RIAGENDR)
    + C(RIDRETH3)
    """,
    data=model_data
).fit()

print(model1.summary())

#Calculate adjusted OR ratio

In [ ]:
import numpy as np

adjusted_OR = np.exp(model1.params['diabetes_combined'])

CI = np.exp(
    model1.conf_int().loc['diabetes_combined']
)

p_value = model1.pvalues['diabetes_combined']

print(f"Adjusted OR: {adjusted_OR:.3f}")
print(f"95% CI: [{CI.iloc[0]:.3f}, {CI.iloc[1]:.3f}]")
print(f"P-value: {p_value:.6f}")

In [ ]:
# Consider Weighting as a relevant variable

In [ ]:
weight_cols = [
    'WTMEC2YR',
    'WTMECPRP',
    'WTINTPRP',
    'WTSAFPRP',
    'SDMVSTRA',
    'SDMVPSU'
]

for col in weight_cols:
    if col in df_adult.columns:
        print(col, ":", df_adult[col].notna().sum(), "non-missing")
    else:
        print(col, ": NOT FOUND")

In [ ]:
# Current analysis sample
analysis_data = df_adult[
    ['diabetes_combined', 'gallstone',
     'WTMECPRP', 'WTSAFPRP',
     'SDMVSTRA', 'SDMVPSU']
].dropna(subset=['diabetes_combined', 'gallstone'])

print("Current sample:", len(analysis_data))

# Participants with valid fasting subsample weights
fasting_sample = analysis_data[
    analysis_data['WTSAFPRP'].notna()
    & (analysis_data['WTSAFPRP'] > 0)
]

print("Fasting subsample:", len(fasting_sample))

# Participants with valid MEC weights
mec_sample = analysis_data[
    analysis_data['WTMECPRP'].notna()
    & (analysis_data['WTMECPRP'] > 0)
]

print("MEC sample:", len(mec_sample))

In [ ]:
fasting_check = df_adult[
    ['diabetes_combined', 'gallstone',
     'LBXGLU', 'WTSAFPRP']
].dropna(subset=['diabetes_combined', 'gallstone'])

fasting_check = fasting_check[
    fasting_check['WTSAFPRP'].notna()
    & (fasting_check['WTSAFPRP'] > 0)
]

print("Fasting subsample:", len(fasting_check))

print("With fasting glucose result:",
      fasting_check['LBXGLU'].notna().sum())

print("Missing fasting glucose:",
      fasting_check['LBXGLU'].isna().sum())

In [ ]:
model_check = df_adult.loc[model_data.index]

print("Model sample:", len(model_check))

print("Missing HbA1c:",
      model_check['LBXGH'].isna().sum())

print("Missing Fasting Glucose:",
      model_check['LBXGLU'].isna().sum())

print("Missing either test:",
      model_check[['LBXGH', 'LBXGLU']].isna().any(axis=1).sum())

print("Both tests available:",
      model_check[['LBXGH', 'LBXGLU']].notna().all(axis=1).sum())

#Complete-case sensitivity analysis（完整病例敏感性分析）

In [ ]:
# Participants with both laboratory measurements
complete_data = df_adult[
    ['gallstone', 'diabetes_combined',
     'RIDAGEYR', 'RIAGENDR', 'RIDRETH3',
     'LBXGH', 'LBXGLU',
     'WTSAFPRP', 'SDMVSTRA', 'SDMVPSU']
].dropna(subset=[
    'gallstone', 'diabetes_combined',
    'RIDAGEYR', 'RIAGENDR', 'RIDRETH3',
    'LBXGH', 'LBXGLU'
]).copy()

print("Complete-case sample:", len(complete_data))

print("Valid fasting weights:",
      (complete_data['WTSAFPRP'] > 0).sum())

In [ ]:
# Model 1 using complete cases
model1_complete = smf.logit(
    formula="""
    gallstone ~ diabetes_combined
    + RIDAGEYR
    + C(RIAGENDR)
    + C(RIDRETH3)
    """,
    data=complete_data
).fit()

# Extract adjusted OR
OR_complete = np.exp(
    model1_complete.params['diabetes_combined']
)

CI_complete = np.exp(
    model1_complete.conf_int().loc['diabetes_combined']
)

print(f"Sample size: {len(complete_data)}")
print(f"Adjusted OR: {OR_complete:.3f}")
print(f"95% CI: [{CI_complete.iloc[0]:.3f}, {CI_complete.iloc[1]:.3f}]")

In [ ]:
print(complete_data['diabetes_combined'].value_counts())

print(pd.crosstab(
    complete_data['diabetes_combined'],
    complete_data['gallstone'],
    margins=True
))

In [ ]:
import numpy as np

for diabetes_status in [0, 1]:

    group = complete_data[
        complete_data['diabetes_combined'] == diabetes_status
    ]

    weighted_prevalence = np.average(
        group['gallstone'],
        weights=group['WTSAFPRP']
    ) * 100

    print(
        f"Diabetes = {diabetes_status}: "
        f"{weighted_prevalence:.2f}%"
    )

#Survey weighiting，Model 1

- 考虑复杂抽样：加入抽样权重、分层和 PSU，使估计及其不确定性考虑 NHANES 的抽样方式。
- 控制混杂因素：控制年龄、性别和族裔，尚未控制吸烟、饮酒。

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
if (!requireNamespace("survey", quietly = TRUE)) install.packages("survey", repos="https://cloud.r-project.org")

In [ ]:
%R -i complete_data

In [ ]:
%%R
library(survey)

design = svydesign(
    ids = ~SDMVPSU,
    strata = ~SDMVSTRA,
    weights = ~WTSAFPRP,
    nest = TRUE,
    data = complete_data
)

In [ ]:
%%R
model1_weighted = svyglm(
    gallstone ~ diabetes_combined +
        RIDAGEYR +
        factor(RIAGENDR) +
        factor(RIDRETH3),
    design = design,
    family = quasibinomial()
)

summary(model1_weighted)

In [ ]:
%%R
beta = coef(model1_weighted)["diabetes_combined"]
se = sqrt(vcov(model1_weighted)["diabetes_combined",
                                "diabetes_combined"])

OR = exp(beta)
CI = exp(beta + c(-1, 1) * 1.96 * se)

cat("Weighted Adjusted OR:", round(OR, 3), "\n")
cat("95% CI:", round(CI, 3), "\n")

#Model 2

### 加入smoking和alcohol变量

### Download smoking daataset(P_SMQ) and alcohol use dataset(P_ALU)

In [ ]:
import pandas as pd

smq = pd.read_sas(folder +"P_SMQ.xpt", format="xport")
alq = pd.read_sas(folder +"P_ALQ.xpt", format="xport")

print("Smoking:", smq.shape)
print("Alcohol:", alq.shape)

print("\nSmoking columns:")
print(smq.columns.tolist())

print("\nAlcohol columns:")
print(alq.columns.tolist())

#Store the files

In [ ]:
df.to_pickle(folder + "df.pkl")
df_adult.to_pickle(folder + "df_adult.pkl")
complete_data.to_pickle(folder + "complete_data.pkl")

print("Saved successfully!")

In [ ]:
# Smoking: ever smoked 100 cigarettes, current smoking frequency
for col in ["SMQ020", "SMQ040"]:
    print(f"\nSmoking — {col}")
    print(smq[col].value_counts(dropna=False).sort_index())

# Alcohol: ever drank alcohol, drinking frequency in past 12 months
for col in ["ALQ111", "ALQ121"]:
    print(f"\nAlcohol — {col}")
    print(alq[col].value_counts(dropna=False).sort_index())

In [ ]:
import numpy as np
import pandas as pd

# 1. Smoking: 0 = never, 1 = former, 2 = current
smoking = smq[["SEQN", "SMQ020", "SMQ040"]].copy()
smoking["smoking_status"] = np.nan

smoking.loc[
    smoking["SMQ020"].eq(2),
    "smoking_status"
] = 0

smoking.loc[
    smoking["SMQ020"].eq(1) & smoking["SMQ040"].eq(3),
    "smoking_status"
] = 1

smoking.loc[
    smoking["SMQ020"].eq(1) & smoking["SMQ040"].isin([1, 2]),
    "smoking_status"
] = 2

# 2. Alcohol: 0 = no drinking in past year, 1 = drinking
alcohol = alq[["SEQN", "ALQ111", "ALQ121"]].copy()

# Correct the near-zero representation in this categorical variable only
near_zero = (
    alcohol["ALQ121"].notna()
    & alcohol["ALQ121"].abs().lt(1e-10)
)
alcohol.loc[near_zero, "ALQ121"] = 0

alcohol["alcohol_current"] = np.nan

alcohol.loc[
    alcohol["ALQ111"].eq(2)
    | (
        alcohol["ALQ111"].eq(1)
        & alcohol["ALQ121"].eq(0)
    ),
    "alcohol_current"
] = 0

alcohol.loc[
    alcohol["ALQ111"].eq(1)
    & alcohol["ALQ121"].isin(range(1, 11)),
    "alcohol_current"
] = 1

# 3. Recover SEQN for the existing complete-case sample
# complete_data kept the original df_adult index
assert df_adult.index.is_unique
assert complete_data.index.is_unique

base = complete_data.copy()
base["SEQN"] = df_adult.loc[base.index, "SEQN"]

model2_candidates = (
    base
    .merge(
        smoking[["SEQN", "smoking_status"]],
        on="SEQN",
        how="left",
        validate="one_to_one"
    )
    .merge(
        alcohol[["SEQN", "alcohol_current"]],
        on="SEQN",
        how="left",
        validate="one_to_one"
    )
)

# 4. Check missingness, then retain complete Model 2 cases
print("Starting sample:", len(model2_candidates))
print("\nMissing new variables:")
print(
    model2_candidates[
        ["smoking_status", "alcohol_current"]
    ].isna().sum()
)

model2_data = model2_candidates.dropna(
    subset=["smoking_status", "alcohol_current"]
).copy()

assert (
    model2_data["WTSAFPRP"].notna()
    & model2_data["WTSAFPRP"].gt(0)
).all()

print("\nModel 2 sample:", len(model2_data))
print("\nSmoking: 0=never, 1=former, 2=current")
print(model2_data["smoking_status"].value_counts().sort_index())
print("\nAlcohol: 0=no drinking in past year, 1=drinking")
print(model2_data["alcohol_current"].value_counts().sort_index())

In [ ]:
%R -i model2_data

In [ ]:
%%R
library(survey)

design2 = svydesign(
    ids = ~SDMVPSU,
    strata = ~SDMVSTRA,
    weights = ~WTSAFPRP,
    nest = TRUE,
    data = model2_data
)

# Model 1, using the same participants as Model 2
model1_same_sample = svyglm(
    gallstone ~ diabetes_combined +
        RIDAGEYR +
        factor(RIAGENDR) +
        factor(RIDRETH3),
    design = design2,
    family = quasibinomial()
)

# Model 2: additionally adjust for smoking and alcohol
model2_weighted = svyglm(
    gallstone ~ diabetes_combined +
        RIDAGEYR +
        factor(RIAGENDR) +
        factor(RIDRETH3) +
        factor(smoking_status) +
        factor(alcohol_current),
    design = design2,
    family = quasibinomial()
)

# Report diabetes OR with t-based confidence intervals
extract_result = function(model, label) {
    term = "diabetes_combined"
    beta = coef(model)[term]
    se = sqrt(vcov(model)[term, term])
    df = model$df.residual

    if (!is.finite(df) || df <= 0) {
        stop("Insufficient residual survey degrees of freedom.")
    }

    critical = qt(0.975, df = df)
    ci = exp(beta + c(-1, 1) * critical * se)
    p = 2 * pt(abs(beta / se), df = df, lower.tail = FALSE)

    data.frame(
        Model = label,
        N = nobs(model),
        OR = unname(exp(beta)),
        Lower95 = ci[1],
        Upper95 = ci[2],
        P_value = unname(p)
    )
}

results = rbind(
    extract_result(model1_same_sample, "Model 1: same sample"),
    extract_result(model2_weighted, "Model 2: + smoking/alcohol")
)

print(results, row.names = FALSE, digits = 4)

#Model 3(增加BMI和贫困收入比）


### Import BMI file


In [ ]:
import pandas as pd

bmx = pd.read_sas(
    folder + "P_BMX.xpt",
    format="xport"
)

print(bmx[["SEQN", "BMXBMI"]].head())

In [ ]:
# 从 Model 2 的样本出发，合并新增变量
model3_candidates = (
    model2_data
    .merge(
        bmx[["SEQN", "BMXBMI"]],
        on="SEQN",
        how="left",
        validate="one_to_one"
    )
    .merge(
        demo[["SEQN", "INDFMPIR"]],
        on="SEQN",
        how="left",
        validate="one_to_one"
    )
)

print("Model 2 sample:", len(model2_data))

print("\nMissing new variables:")
print(model3_candidates[["BMXBMI", "INDFMPIR"]].isna().sum())

# 保留两个新增变量都有数据的人
model3_data = model3_candidates.dropna(
    subset=["BMXBMI", "INDFMPIR"]
).copy()

print("\nModel 3 sample:", len(model3_data))
print(model3_data[["BMXBMI", "INDFMPIR"]].describe())

In [ ]:
# 将收入比中接近零的数值恢复为 0
near_zero = model3_data["INDFMPIR"].abs().lt(1e-10)
model3_data.loc[near_zero, "INDFMPIR"] = 0

# 传给 R
%R -i model3_data

In [ ]:
%%R
library(survey)

design3 = svydesign(
    ids = ~SDMVPSU,
    strata = ~SDMVSTRA,
    weights = ~WTSAFPRP,
    nest = TRUE,
    data = model3_data
)

# Model 2：在 Model 3 的同一批人上重新拟合
model2_same_sample = svyglm(
    gallstone ~ diabetes_combined +
        RIDAGEYR +
        factor(RIAGENDR) +
        factor(RIDRETH3) +
        factor(smoking_status) +
        factor(alcohol_current),
    design = design3,
    family = quasibinomial()
)

# Model 3：额外控制 BMI 和贫困收入比
model3_weighted = svyglm(
    gallstone ~ diabetes_combined +
        RIDAGEYR +
        factor(RIAGENDR) +
        factor(RIDRETH3) +
        factor(smoking_status) +
        factor(alcohol_current) +
        BMXBMI +
        INDFMPIR,
    design = design3,
    family = quasibinomial()
)

# 使用前面定义过的结果提取函数
results3 = rbind(
    extract_result(model2_same_sample, "Model 2: same sample"),
    extract_result(model3_weighted, "Model 3: + BMI/PIR")
)

print(results3, row.names = FALSE, digits = 4)

#孟德尔随机化

In [ ]:
from pathlib import Path

# Frozen, harmonized summary statistics for 109 SNPs.
(DATA_DIR / "T2D_gallstone_harmonized.csv").write_text(
    """CHR,BP,SNP,A1,A2,frq_A1,b,se,P,N,F_stat,beta_gallstone,se_gallstone,effect_allele_gallstone,other_allele_gallstone,eaf_gallstone,p_gallstone,harmonization_status,beta_sign,mr_keep,beta_gallstone_aligned,eaf_gallstone_aligned
10,114758349,rs7903146,T,C,0.291584512494863,0.3059,0.0077,0.0,597475,1578.256198347107,-0.0069,0.015,T,C,0.1996,0.644401,直接对齐,1.0,True,-0.0069,0.1996
6,20679709,rs7756992,G,A,0.266895680827625,0.1297,0.0078,5.999e-62,605054,276.4972057856674,0.0114,0.0127,G,A,0.3327,0.3694,直接对齐,1.0,True,0.0114,0.3327
9,22134094,rs10811661,C,T,0.173605396001198,-0.1569,0.0098,4.132000000000001e-58,605005,256.3266347355269,-0.0046,0.017,C,T,0.144,0.7879,直接对齐,1.0,True,-0.0046,0.144
3,185513392,rs7651090,G,A,0.313376825056666,0.1204,0.0076,3.854e-57,605051,250.97229916897504,0.0109,0.0129,G,A,0.3077,0.4005,直接对齐,1.0,True,0.0109,0.3077
8,118185025,rs3802177,A,G,0.311281439217812,-0.1217,0.008,2.321e-52,596090,231.42015625,-0.0228,0.0124,A,G,0.3762,0.0647202,直接对齐,1.0,True,-0.0228,0.3762
16,53822651,rs7185735,G,A,0.397306062260211,0.1056,0.0073,1.59e-47,597339,209.2580221429912,0.0411,0.0122,G,A,0.4028,0.000756502,直接对齐,1.0,True,0.0411,0.4028
7,28196413,rs849135,G,A,0.500947820257996,0.0999,0.0072,1.041e-43,597276,192.515625,-0.0077,0.012,A,G,0.4631,0.5239,等位基因互换，beta 反号,-1.0,True,0.0077,0.5368999999999999
10,94482076,rs7923866,T,C,0.378774900654434,-0.0972,0.0074,9.337e-40,604875,172.53177501826147,-0.0071,0.0122,T,C,0.3972,0.563101,直接对齐,1.0,True,-0.0071,0.3972
2,227101411,rs2972144,A,G,0.354633108192389,-0.0913,0.0075,2.5510000000000003e-34,604129,148.19004444444445,0.0022,0.0125,G,A,0.6386,0.8573,等位基因互换，beta 反号,-1.0,True,-0.0022,0.36140000000000005
4,6303022,rs1801214,C,T,0.400415373416361,-0.0903,0.0074,5.5160000000000005e-34,596870,148.90595325054787,-0.0085,0.0122,T,C,0.5763,0.4859,等位基因互换，beta 反号,-1.0,True,0.0085,0.42369999999999997
3,123065778,rs11708067,G,A,0.238989658490694,-0.0965,0.0086,5.933e-29,604949,125.90927528393723,-0.0361,0.016,G,A,0.1694,0.0239398,直接对齐,1.0,True,-0.0361,0.1694
2,43707385,rs17334919,T,C,0.100217908588586,-0.1398,0.0128,6.687e-28,604236,119.287353515625,0.0447,0.028,T,C,0.04824,0.1096,直接对齐,1.0,True,0.0447,0.04824
11,72433098,rs1552224,C,A,0.15421045285969,-0.1034,0.0101,8.635e-25,597470,104.8089402999706,0.0058,0.0141,C,A,0.2369,0.682801,直接对齐,1.0,True,0.0058,0.2369
3,12394840,rs1899951,T,C,0.123288181623979,-0.1118,0.0109,1.637e-24,604718,105.20360239037116,-0.0244,0.0159,T,C,0.1697,0.1249,直接对齐,1.0,True,-0.0244,0.1697
13,80717156,rs1359790,A,G,0.286699863937315,-0.0796,0.008,2.795e-23,605054,99.00250000000004,-0.0044,0.0132,A,G,0.2935,0.737099,直接对齐,1.0,True,-0.0044,0.2935
9,84308948,rs2796441,A,G,0.416458049639807,-0.0715,0.0073,1.962e-22,602118,95.93263276412084,0.0104,0.0121,A,G,0.4176,0.3917,直接对齐,1.0,True,0.0104,0.4176
2,165528876,rs13389219,T,C,0.394367937714495,-0.0722,0.0074,2.106e-22,597298,95.19430241051862,0.0141,0.0126,T,C,0.3479,0.2645,直接对齐,1.0,True,0.0141,0.3479
8,41519248,rs516946,T,C,0.23936670103039,-0.0824,0.0085,3.159e-22,605047,93.97591695501728,-0.0104,0.0149,C,T,0.796,0.4829,等位基因互换，beta 反号,-1.0,True,0.0104,0.20399999999999996
17,36099840,rs11651755,C,T,0.484669254001265,0.0741,0.0077,8.978999999999998e-22,557434,92.60937763535166,-0.0103,0.0123,T,C,0.6173,0.4059,等位基因互换，beta 反号,-1.0,True,0.0103,0.38270000000000004
2,27741237,rs780094,T,C,0.387154700626533,-0.0692,0.0074,5.159e-21,605021,87.44777209642074,0.0744,0.0126,C,T,0.6432,3.25897e-09,等位基因互换，beta 反号,-1.0,True,-0.0744,0.3568
11,17408630,rs5215,C,T,0.360058778070131,0.0678,0.0073,2.0890000000000003e-20,605049,86.26083693000561,0.0002,0.012,T,C,0.5283,0.9843,等位基因互换，beta 反号,-1.0,True,-0.0002,0.4717
7,15064255,rs2191348,G,T,0.45306459897734,-0.0652,0.0073,3.444e-19,594267,79.77181459936195,-0.0146,0.012,T,G,0.4597,0.2268,等位基因互换，beta 反号,-1.0,True,0.0146,0.5403
6,32627714,rs1063355,T,G,0.397563657636028,-0.0709,0.0079,3.715e-19,567291,80.54494472039737,0.0058,0.0129,G,T,0.526,0.652299,等位基因互换，beta 反号,-1.0,True,-0.0058,0.474
3,23454790,rs1496653,G,A,0.204781842750379,-0.0769,0.0088,2.572e-18,604458,76.36376549586777,-0.0088,0.013,G,A,0.3119,0.4968,直接对齐,1.0,True,-0.0088,0.3119
1,214159256,rs340874,T,C,0.43609575454678,-0.0626,0.0073,8.406e-18,597473,73.53649840495402,-0.0098,0.0121,C,T,0.4315,0.4189,等位基因互换，beta 反号,-1.0,True,0.0098,0.5685
5,55806751,rs459193,A,G,0.254662376595439,-0.0711,0.0083,8.809e-18,597479,73.38089708230513,-0.0435,0.0129,G,A,0.683,0.000720195,等位基因互换，beta 反号,-1.0,True,0.0435,0.31699999999999995
5,102100576,rs7729395,T,C,0.0509409339093332,0.1373,0.016,1.101e-17,597473,73.63785156250002,0.013,0.0262,T,C,0.05587,0.618899,直接对齐,1.0,True,0.013,0.05587
10,12307894,rs11257655,T,C,0.206773280772991,0.0737,0.0087,1.966e-17,597480,71.76231998943057,-0.0103,0.0137,T,C,0.2649,0.4539,直接对齐,1.0,True,-0.0103,0.2649
12,66212318,rs2261181,T,C,0.096470032992502,0.0985,0.0118,9.179e-17,600965,69.68004883654125,-0.0077,0.0232,T,C,0.07334,0.739301,直接对齐,1.0,True,-0.0077,0.07334
12,27965150,rs10842994,T,C,0.197025394898452,-0.0755,0.0091,1.015e-16,605053,68.83528559352733,-0.0084,0.0159,T,C,0.1731,0.5948,直接对齐,1.0,True,-0.0084,0.1731
15,77832762,rs7177055,G,A,0.28171133230901,-0.0647,0.0079,2.746e-16,605052,67.07402659830154,0.0214,0.013,A,G,0.6932,0.1004,等位基因互换，beta 反号,-1.0,True,-0.0214,0.30679999999999996
19,46158513,rs8108269,G,T,0.281015346375071,0.0644,0.0079,3.114e-16,604649,66.45345297228006,-0.0091,0.0129,G,T,0.3129,0.4831,直接对齐,1.0,True,-0.0091,0.3129
7,14898282,rs17168486,T,C,0.173603116062419,0.0742,0.0094,2.177e-15,592191,62.30918967858759,0.0204,0.0151,T,C,0.198,0.1758,直接对齐,1.0,True,0.0204,0.198
2,60585806,rs243019,C,T,0.455830717095865,0.0566,0.0071,2.29e-15,599688,63.55008926800237,-0.0182,0.0121,C,T,0.4485,0.1314,直接对齐,1.0,True,-0.0182,0.4485
9,4291928,rs10974438,C,A,0.351215240515481,0.0591,0.0075,3.013e-15,593472,62.0944,-0.0133,0.0124,C,A,0.3759,0.2822,直接对齐,1.0,True,-0.0133,0.3759
3,64705365,rs6795735,T,C,0.410911492715474,-0.0558,0.0073,1.63e-14,605050,58.42822293113155,0.0001,0.0125,T,C,0.3607,0.9923,直接对齐,1.0,True,0.0001,0.3607
12,121456616,rs61953351,T,G,0.249902347563897,-0.07,0.0091,1.976e-14,572951,59.171597633136095,0.0271,0.0133,T,G,0.2836,0.0421697,直接对齐,1.0,True,0.0271,0.2836
17,3985864,rs8068804,A,G,0.325096537247602,0.0587,0.0078,4.411e-14,588147,56.63527284681132,0.0048,0.0131,A,G,0.2986,0.7141,直接对齐,1.0,True,0.0048,0.2986
1,39913351,rs2296173,G,A,0.212010575612847,0.065,0.0087,7.658e-14,597394,55.8197912537984,-0.0074,0.0153,G,A,0.1907,0.6297,直接对齐,1.0,True,-0.0074,0.1907
17,46989154,rs9894220,G,A,0.433739992660085,-0.0585,0.0079,1.517e-13,573700,54.83496234577791,0.0013,0.0122,G,A,0.4103,0.9157,直接对齐,1.0,True,0.0013,0.4103
12,124568456,rs825476,C,T,0.41945071095672,-0.0524,0.0073,6.805e-13,597014,51.524863951960974,0.0101,0.0122,T,C,0.5952,0.4082,等位基因互换，beta 反号,-1.0,True,-0.0101,0.40480000000000005
5,76427311,rs6878122,G,A,0.318208634557753,0.0564,0.0079,1.188e-12,592977,50.96875500721037,0.0066,0.0143,A,G,0.7723,0.6427,等位基因互换，beta 反号,-1.0,True,-0.0066,0.2277
1,219660535,rs2820426,A,G,0.389942484385998,-0.0521,0.0073,1.302e-12,597937,50.93657346594107,0.0132,0.0123,G,A,0.6135,0.2849,等位基因互换，beta 反号,-1.0,True,-0.0132,0.38649999999999995
15,91511260,rs12910825,G,A,0.360390808672187,0.0517,0.0074,2.164e-12,605051,48.81099342585829,0.0024,0.0124,G,A,0.3689,0.8441,直接对齐,1.0,True,0.0024,0.3689
9,81898980,rs17791483,G,A,0.062588322668909,-0.102,0.0147,3.419e-12,597198,48.14660558100791,0.0073,0.0184,G,A,0.1211,0.691399,直接对齐,1.0,True,0.0073,0.1211
19,19407718,rs10401969,C,T,0.0765857554285534,0.0921,0.0133,4.131e-12,604393,47.95302165187406,0.0013,0.0245,C,T,0.06377,0.9586,直接对齐,1.0,True,0.0013,0.06377
18,57884750,rs12970134,A,G,0.265120385616101,0.0555,0.008,5.309e-12,602401,48.12890625,0.0288,0.0151,A,G,0.1953,0.0555802,直接对齐,1.0,True,0.0288,0.1953
6,127068983,rs853974,T,C,0.26241384700306,0.0601,0.0088,7.858e-12,571457,46.64269111570248,0.0069,0.0135,C,T,0.7315,0.612099,等位基因互换，beta 反号,-1.0,True,-0.0069,0.26849999999999996
16,81534790,rs2925979,T,C,0.299149855500384,0.0534,0.0078,9.061e-12,596099,46.86982248520711,0.016,0.0129,C,T,0.6776,0.2137,等位基因互换，beta 反号,-1.0,True,-0.016,0.3224
1,51506886,rs12088739,G,A,0.089848136244033,-0.0884,0.013,9.792e-12,596436,46.24000000000001,0.0185,0.0252,G,A,0.05914,0.4625,直接对齐,1.0,True,0.0185,0.05914
5,53272664,rs4865796,G,A,0.306909895697633,-0.053,0.0078,1.329e-11,597478,46.170282708744246,0.0129,0.0127,A,G,0.667,0.3103,等位基因互换，beta 反号,-1.0,True,-0.0129,0.33299999999999996
2,161136656,rs7572970,A,G,0.277953371585924,-0.059,0.0087,1.391e-11,577003,45.99022327916502,0.0199,0.0149,G,A,0.796,0.1803,等位基因互换，beta 反号,-1.0,True,-0.0199,0.20399999999999996
13,33554302,rs576674,G,A,0.167485416105361,0.0654,0.0097,1.792e-11,594299,45.45817833988734,-0.0073,0.0205,A,G,0.9059,0.722199,等位基因互换，beta 反号,-1.0,True,0.0073,0.09409999999999996
4,185716232,rs735949,C,T,0.141149742372597,-0.0711,0.0106,1.946e-11,597370,44.991189035243856,0.0161,0.0199,C,T,0.1021,0.4183,直接对齐,1.0,True,0.0161,0.1021
10,80964975,rs753270,T,C,0.416465397344019,-0.0528,0.0079,2.702e-11,571222,44.66976446082357,0.0012,0.0122,C,T,0.5783,0.9203,等位基因互换，beta 反号,-1.0,True,-0.0012,0.42169999999999996
2,121309759,rs12617659,T,C,0.147238129122417,-0.0685,0.0103,2.827e-11,601859,44.22895654632859,0.0137,0.0168,T,C,0.1486,0.4158,直接对齐,1.0,True,0.0137,0.1486
6,7281654,rs1050226,G,A,0.406792300585879,-0.0491,0.0074,3.342e-11,593598,44.025018261504734,0.0105,0.0124,G,A,0.3762,0.3967,直接对齐,1.0,True,0.0105,0.3762
3,187740899,rs4686471,T,C,0.390224945319632,-0.0534,0.0081,4.282e-11,564615,43.462277091906735,-0.0007,0.0123,C,T,0.6092,0.9578,等位基因互换，beta 反号,-1.0,True,0.0007,0.39080000000000004
6,50816887,rs72892910,T,G,0.172394287519017,0.0648,0.0099,6.428e-11,562619,42.84297520661156,0.0098,0.0149,T,G,0.2029,0.512,直接对齐,1.0,True,0.0098,0.2029
6,43810974,rs9369425,G,A,0.291815284019485,0.0546,0.0085,1.129e-10,578544,41.26173010380623,0.0321,0.013,A,G,0.6885,0.0132501,等位基因互换，beta 反号,-1.0,True,-0.0321,0.3115
1,120471224,rs2493394,G,A,0.107337868211843,0.073,0.0113,1.151e-10,594129,41.733886756989584,-0.0216,0.0174,G,A,0.1369,0.2146,直接对齐,1.0,True,-0.0216,0.1369
12,4406281,rs12299509,G,A,0.478621745294057,0.0467,0.0073,2.087e-10,592283,40.92493901294801,-0.0085,0.0123,G,A,0.5067,0.4899,直接对齐,1.0,True,-0.0085,0.5067
4,153254121,rs7685296,T,C,0.279364611974842,-0.0511,0.0081,2.317e-10,596473,39.79896357262613,-0.0399,0.0132,T,C,0.2934,0.00253402,直接对齐,1.0,True,-0.0399,0.2934
15,63823301,rs982077,A,G,0.434479031724371,0.0453,0.0072,2.577e-10,602960,39.58506944444445,-0.0112,0.0121,G,A,0.4329,0.3564,等位基因互换，beta 反号,-1.0,True,0.0112,0.5670999999999999
7,117510621,rs13239186,T,C,0.302028595537338,0.0539,0.0085,2.704e-10,562451,40.21051903114186,-0.0009,0.0125,T,C,0.3665,0.9411,直接对齐,1.0,True,-0.0009,0.3665
6,160766770,rs622217,C,T,0.483931554092884,-0.0485,0.0077,3.129e-10,558700,39.673638050261424,0.0163,0.012,C,T,0.5023,0.1735,直接对齐,1.0,True,0.0163,0.5023
22,44377442,rs4823182,G,A,0.33574764315569,0.0482,0.0077,3.358e-10,587224,39.18434811941305,-0.0181,0.0121,G,A,0.4431,0.1337,直接对齐,1.0,True,-0.0181,0.4431
2,622827,rs2867125,T,C,0.172175281510457,-0.0601,0.0096,4.325e-10,595791,39.19281684027778,0.024,0.0161,C,T,0.8357,0.1373,等位基因互换，beta 反号,-1.0,True,-0.024,0.1643
4,83584496,rs993380,A,G,0.334449877168873,0.0507,0.0081,4.587e-10,579176,39.17832647462277,0.0264,0.0122,G,A,0.5958,0.0309699,等位基因互换，beta 反号,-1.0,True,-0.0264,0.4042
11,128398938,rs67232546,T,C,0.20922179282714,0.0596,0.0096,4.661e-10,563610,38.543402777777786,-0.0013,0.0168,T,C,0.1517,0.9365,直接对齐,1.0,True,-0.0013,0.1517
2,65701757,rs840967,C,A,0.394078203325672,0.0497,0.008,5.442e-10,570367,38.59515625,-0.0103,0.012,A,C,0.4809,0.3924,等位基因互换，beta 反号,-1.0,True,0.0103,0.5191
12,71439589,rs7138300,C,T,0.44316503609743,0.0443,0.0072,5.649e-10,601951,37.85667438271605,0.004,0.0124,T,C,0.6293,0.746799,等位基因互换，beta 反号,-1.0,True,-0.004,0.37070000000000003
4,103988899,rs7674212,T,G,0.408864361771549,-0.0465,0.0075,6.182e-10,576752,38.440000000000005,-0.0028,0.0122,T,G,0.4251,0.8153,直接对齐,1.0,True,-0.0028,0.4251
8,10633159,rs10100265,A,C,0.389510131879901,0.0491,0.0079,6.288e-10,594125,38.62858516263419,-0.0434,0.0164,C,A,0.542,0.00797297,等位基因互换，beta 反号,-1.0,True,0.0434,0.45799999999999996
11,2839751,rs2237892,T,C,0.0624895946466012,-0.096,0.0157,8.746e-10,594811,37.38894072781858,0.0039,0.0272,T,C,0.05139,0.886,直接对齐,1.0,True,0.0039,0.05139
16,300641,rs9940149,A,G,0.17855594571223,-0.058,0.0095,9.291e-10,605054,37.27423822714682,-0.0181,0.0194,A,G,0.1059,0.3514,直接对齐,1.0,True,-0.0181,0.1059
6,31478986,rs2246618,T,C,0.307253342139738,0.0513,0.0084,1.203e-09,573704,37.297193877551024,-0.0037,0.014,T,C,0.296,0.792801,直接对齐,1.0,True,-0.0037,0.296
17,17653411,rs12945601,T,C,0.386396556423368,0.048,0.008,1.718e-09,572260,36.0,0.0069,0.0121,C,T,0.5637,0.5697,等位基因互换，beta 反号,-1.0,True,-0.0069,0.4363
1,229672955,rs348330,G,A,0.366548876273864,0.0487,0.0081,1.864e-09,568792,36.14830056393843,-0.0023,0.0126,A,G,0.6446,0.8563,等位基因互换，beta 反号,-1.0,True,0.0023,0.35540000000000005
20,45594711,rs6066138,A,G,0.278361954233567,-0.049,0.0082,1.929e-09,595265,35.70791195716835,-0.0318,0.0146,A,G,0.2147,0.0297797,直接对齐,1.0,True,-0.0318,0.2147
11,49351026,rs7929543,C,A,0.0831598873051136,0.0828,0.0138,2.199e-09,579516,36.0,-0.0112,0.024,C,A,0.07541,0.6402,直接对齐,1.0,True,-0.0112,0.07541
17,40542501,rs17405722,A,G,0.0741526485278786,0.087,0.0146,2.278e-09,573202,35.50853818727715,0.0286,0.02,A,G,0.1004,0.1538,直接对齐,1.0,True,0.0286,0.1004
8,30863722,rs10087241,G,A,0.405106648266454,0.0475,0.008,2.796e-09,565916,35.25390625,-0.0088,0.0125,A,G,0.6389,0.4814,等位基因互换，beta 反号,-1.0,True,0.0088,0.3611
5,14847331,rs1061813,G,A,0.462880801655686,0.0429,0.0073,3.372e-09,593583,34.5357477950835,-0.012,0.0123,A,G,0.6192,0.3321,等位基因互换，beta 反号,-1.0,True,0.012,0.3808
7,44248828,rs2908282,A,G,0.177394802964819,0.0552,0.0094,4.25e-09,604544,34.48438207333635,0.0041,0.0194,A,G,0.1071,0.8345,直接对齐,1.0,True,0.0041,0.1071
8,95937502,rs7845219,C,T,0.49278585092858,-0.0422,0.0072,4.545e-09,595747,34.35262345679013,-0.0087,0.012,C,T,0.5103,0.4655,直接对齐,1.0,True,-0.0087,0.5103
22,30552813,rs16988333,G,A,0.09040354926268,-0.0745,0.013,9.169e-09,600076,32.841715976331365,-0.008,0.0215,G,A,0.08541,0.7112,直接对齐,1.0,True,-0.008,0.08541
7,156983847,rs7786095,G,A,0.103859860937899,-0.0743,0.0129,9.643e-09,579310,33.17402800312482,0.0062,0.0256,G,A,0.05743,0.808,直接对齐,1.0,True,0.0062,0.05743
9,96919182,rs10114341,C,T,0.440754140705696,-0.0409,0.0072,1.151e-08,602097,32.26871141975308,0.0211,0.0121,C,T,0.4235,0.0817692,直接对齐,1.0,True,0.0211,0.4235
3,63897215,rs2292662,T,C,0.151271963500907,-0.0629,0.0111,1.236e-08,572779,32.11111111111111,-0.0049,0.0146,T,C,0.2174,0.734999,直接对齐,1.0,True,-0.0049,0.2174
7,103418846,rs2299383,T,C,0.423455221755939,0.0412,0.0073,1.49e-08,590541,31.852880465378124,-0.0036,0.012,T,C,0.4654,0.764999,直接对齐,1.0,True,-0.0036,0.4654
20,32675727,rs6059662,A,G,0.336824454484242,-0.0446,0.0079,1.512e-08,599260,31.87245633712546,0.0482,0.0143,G,A,0.769,0.000738992,等位基因互换，beta 反号,-1.0,True,-0.0482,0.23099999999999998
16,69563842,rs13330951,G,A,0.488313929064598,-0.0456,0.0081,1.538e-08,575181,31.692729766803843,-0.0254,0.0121,G,A,0.5003,0.0357998,直接对齐,1.0,True,-0.0254,0.5003
8,146003567,rs2294120,G,A,0.455879299759268,-0.0443,0.0079,1.618e-08,559430,31.445120974202847,-0.0086,0.0121,G,A,0.468,0.4737,直接对齐,1.0,True,-0.0086,0.468
14,79940383,rs7144011,T,G,0.221063370925376,0.0482,0.0085,1.636e-08,604397,32.15557093425605,0.0207,0.014,T,G,0.2398,0.1403,直接对齐,1.0,True,0.0207,0.2398
3,46925539,rs11926707,T,C,0.374443928283118,-0.0463,0.0082,1.686e-08,563896,31.881171921475303,0.0021,0.0127,C,T,0.6523,0.868,等位基因互换，beta 反号,-1.0,True,-0.0021,0.3477
4,123833154,rs11098676,T,C,0.212360864516997,-0.054,0.0096,2.027e-08,573362,31.640625,0.0004,0.0156,C,T,0.8204,0.9793,等位基因互换，beta 反号,-1.0,True,-0.0004,0.17959999999999998
10,71485458,rs10740322,G,A,0.312896190286823,-0.0477,0.0085,2.109e-08,567457,31.491903114186847,0.0118,0.0137,A,G,0.7417,0.3867,等位基因互换，beta 反号,-1.0,True,-0.0118,0.2583
20,43001721,rs4810426,T,C,0.096788799683677,0.0726,0.013,2.148e-08,569858,31.18792899408284,0.0036,0.0172,T,C,0.1446,0.8317,直接对齐,1.0,True,0.0036,0.1446
18,7070642,rs7240767,C,T,0.383677476727202,0.0451,0.0081,2.157e-08,572727,31.001524157902768,-0.0088,0.0123,C,T,0.3951,0.474,直接对齐,1.0,True,-0.0088,0.3951
3,170766618,rs11925227,A,G,0.183438659087805,-0.0534,0.0095,2.25e-08,587754,31.596232686980617,-0.0064,0.0152,A,G,0.1948,0.6743,直接对齐,1.0,True,-0.0064,0.1948
1,117530507,rs1127655,C,T,0.470936253793588,0.0438,0.0079,2.471e-08,571442,30.73930459862201,0.0011,0.012,T,C,0.5361,0.9285,等位基因互换，beta 反号,-1.0,True,-0.0011,0.4639
4,53134293,rs17086692,T,G,0.313425861468874,-0.0467,0.0084,2.481e-08,579149,30.908304988662127,-0.0184,0.0129,T,G,0.3123,0.1541,直接对齐,1.0,True,-0.0184,0.3123
6,131950233,rs3756784,G,T,0.185838123441613,0.0505,0.0091,2.589e-08,594130,30.796401400797,-0.0067,0.0145,G,T,0.2203,0.6418,直接对齐,1.0,True,-0.0067,0.2203
2,228973660,rs7561798,G,A,0.482171024837791,0.04,0.0072,2.794e-08,597003,30.864197530864207,-0.009,0.0121,G,A,0.4513,0.4537,直接对齐,1.0,True,-0.009,0.4513
20,22435749,rs6515236,C,A,0.249330468323155,-0.0504,0.0091,3.343e-08,573439,30.67455621301775,-0.0131,0.0148,C,A,0.2082,0.3757,直接对齐,1.0,True,-0.0131,0.2082
12,93978504,rs11107116,T,G,0.219714310921347,0.0467,0.0085,3.75e-08,605050,30.185328719723174,-0.0101,0.0138,T,G,0.251,0.4651,直接对齐,1.0,True,-0.0101,0.251
17,61687600,rs17631783,T,C,0.263459811367037,-0.0487,0.0089,3.949e-08,574324,29.941800277742708,-0.0031,0.0144,T,C,0.2278,0.8283,直接对齐,1.0,True,-0.0031,0.2278
5,112927686,rs10077431,A,C,0.214668130453798,-0.0487,0.0089,4.755e-08,594019,29.941800277742708,-0.0251,0.0158,A,C,0.175,0.1114,直接对齐,1.0,True,-0.0251,0.175
""", encoding="utf-8"
)


##IVW method






In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

folder = DATA_DIR
df = pd.read_csv(folder / "T2D_gallstone_harmonized.csv")

b_x = df["b"]                           # 糖尿病 beta
b_y = df["beta_gallstone_aligned"]       # 对齐后的胆结石 beta
se_y = df["se_gallstone"]                # 胆结石 SE

valid = (
    np.isfinite(b_x) & np.isfinite(b_y) & np.isfinite(se_y)
    & b_x.ne(0) & se_y.gt(0)
)
if df.empty or not valid.all():
    raise ValueError("请检查输入表是否为空，或是否有缺失值、零糖尿病 beta 或无效的胆结石 SE。")

# 1. 每个 SNP 的效应估计和标准误
df["wald_ratio"] = b_y / b_x
df["wald_se"] = se_y / b_x.abs()

# 2. 权重 = 方差的倒数，也就是 b_x² / se_y²
df["ivw_weight"] = 1 / df["wald_se"]**2

# 3. 将所有 SNP 的效应估计加权平均
total_weight = df["ivw_weight"].sum()
beta_ivw = (df["ivw_weight"] * df["wald_ratio"]).sum() / total_weight
se_ivw = np.sqrt(1 / total_weight)

# 4. 转成 OR，并计算 95% CI 和双侧 P 值
lower_beta = beta_ivw - 1.96 * se_ivw
upper_beta = beta_ivw + 1.96 * se_ivw
p_value = 2 * norm.sf(abs(beta_ivw / se_ivw))

result = pd.DataFrame([{
    "Method": "Fixed-effect IVW",
    "N_SNP": len(df),
    "Beta": beta_ivw,
    "SE": se_ivw,
    "OR": np.exp(beta_ivw),
    "Lower95": np.exp(lower_beta),
    "Upper95": np.exp(upper_beta),
    "P_value": p_value,
}])

display(result)
display(df[["SNP", "wald_ratio", "wald_se", "ivw_weight"]].head())

# Weighted Median Method

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

folder = DATA_DIR
df = pd.read_csv(folder / "T2D_gallstone_harmonized.csv")

b_x = df["b"].to_numpy(dtype=float)                       # 糖尿病 beta
se_x = df["se"].to_numpy(dtype=float)                     # 糖尿病 SE
b_y = df["beta_gallstone_aligned"].to_numpy(dtype=float)  # 对齐后的胆结石 beta
se_y = df["se_gallstone"].to_numpy(dtype=float)           # 胆结石 SE

if len(df) < 3:
    raise ValueError("加权中位数至少需要 3 个 SNP。")
if (
    not np.isfinite([b_x, se_x, b_y, se_y]).all()
    or np.any(b_x == 0)
    or np.any(se_x <= 0)
    or np.any(se_y <= 0)
):
    raise ValueError("请检查 beta 是否缺失、糖尿病 beta 是否为 0，或 SE 是否无效。")

def weighted_median(ratios, weights):
    order = np.argsort(ratios)
    sorted_ratios = ratios[order]
    sorted_weights = weights[order]

    # 给每个比值分配累计权重区间的中点，再插值求 50% 位置
    positions = (np.cumsum(sorted_weights) - 0.5 * sorted_weights)
    positions = positions / sorted_weights.sum()
    return np.interp(0.5, positions, sorted_ratios)

# 1. 算每个 SNP 的 Wald 比值和权重
wald_ratio = b_y / b_x
wald_variance = (se_y / b_x)**2 + (b_y * se_x / b_x**2)**2
weights = 1 / wald_variance

# 2. 找到加权中位数
beta_wm = weighted_median(wald_ratio, weights)

# 3. 按各 SNP 的 beta 和 SE 模拟，使用原始权重估计标准误
rng = np.random.default_rng(42)
n_boot = 1000
boot_x = rng.normal(b_x, se_x, size=(n_boot, len(df)))
boot_y = rng.normal(b_y, se_y, size=(n_boot, len(df)))
boot_estimates = [
    weighted_median(ratios, weights)
    for ratios in boot_y / boot_x
]
se_wm = np.std(boot_estimates, ddof=1)

# 4. 输出 OR、95% 置信区间和双侧 P 值
z = norm.ppf(0.975)
result_wm = pd.DataFrame([{
    "Method": "Weighted median",
    "N_SNP": len(df),
    "Beta": beta_wm,
    "SE": se_wm,
    "OR": np.exp(beta_wm),
    "Lower95": np.exp(beta_wm - z * se_wm),
    "Upper95": np.exp(beta_wm + z * se_wm),
    "P_value": 2 * norm.sf(abs(beta_wm / se_wm)),
}])

display(result_wm)

OR 来自原始数据，置信区间和 P 值用到了模拟得到的 SE。用加权平均值测试1000次后发现se，即分散程或标准误已经大于原始的beta，说明波动已经大于原始效应，模型体现数据没有展现出statistical significance.

Beta = 0.023424 来自原始数据的加权中位数；SE = 0.029502 来自 1000 个模拟中位数的分散程度。

# MR_Egger model

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import t

folder = DATA_DIR
df = pd.read_csv(folder / "T2D_gallstone_harmonized.csv")

b_x = df["b"].to_numpy(dtype=float)
b_y = df["beta_gallstone_aligned"].to_numpy(dtype=float)
se_y = df["se_gallstone"].to_numpy(dtype=float)

if (
    len(df) < 3
    or not np.isfinite([b_x, b_y, se_y]).all()
    or np.any(b_x == 0)
    or np.any(se_y <= 0)
    or np.ptp(np.abs(b_x)) == 0
):
    raise ValueError("请检查 SNP 数量、beta 和胆结石 SE 是否有效。")

# 把效应方向统一到增加糖尿病易感性的等位基因。
# 哪个 SNP 的糖尿病 beta 为负，就把它的两个 beta 同时反号。
x = np.abs(b_x)
y = b_y * np.sign(b_x)

# 带截距的加权回归，权重为 1 / 胆结石 SE²。
design = sm.add_constant(x, has_constant="add")
fit = sm.WLS(y, design, weights=1 / se_y**2).fit()

intercept, beta_egger = fit.params
df_resid = int(fit.df_resid)

# 按 TwoSampleMR 的规则，残差离散度低于 1 时不缩小 SE。
se_intercept, se_egger = np.sqrt(
    np.diag(fit.normalized_cov_params) * max(1.0, fit.scale)
)
critical = t.ppf(0.975, df_resid)

result_egger = pd.DataFrame([{
    "Method": "MR-Egger",
    "N_SNP": len(df),
    "Beta": beta_egger,
    "SE": se_egger,
    "OR": np.exp(beta_egger),
    "Lower95": np.exp(beta_egger - critical * se_egger),
    "Upper95": np.exp(beta_egger + critical * se_egger),
    "P_value": 2 * t.sf(abs(beta_egger / se_egger), df_resid),
}])

egger_intercept = pd.DataFrame([{
    "Intercept": intercept,
    "SE": se_intercept,
    "P_value": 2 * t.sf(abs(intercept / se_intercept), df_resid),
}])

display(result_egger)
display(egger_intercept)

MR-Egger 在加权回归里估计两个数。斜率是糖尿病遗传易感性对胆结石的效应估计，可以转成 OR 和前两个模型比较。截距用来检验这些 SNP 是否存在平均偏向同一方向的水平多效性。假设有些 SNP 除了与糖尿病有关，还通过其他途径影响胆结石，这就叫水平多效性。如果这些额外作用大多把胆结石风险往同一方向推，拟合的直线就可能整体上移或下移。截距检验看的就是这种平均方向性偏移。截距接近零时，仍可能有个别 SNP 存在额外作用，只是正负作用相互抵消了

MR‑Egger 做的是加权线性回归。每个 SNP 的横坐标是糖尿病 beta，纵坐标是胆结石 beta，回归权重由胆结石 beta 的 SE

#简单众数模型


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm, t, median_abs_deviation

folder = DATA_DIR
df = pd.read_csv(folder / "T2D_gallstone_harmonized.csv")

b_x = df["b"].to_numpy(dtype=float)                       # 糖尿病 beta
se_x = df["se"].to_numpy(dtype=float)                     # 糖尿病 SE
b_y = df["beta_gallstone_aligned"].to_numpy(dtype=float)   # 对齐后的胆结石 beta
se_y = df["se_gallstone"].to_numpy(dtype=float)            # 胆结石 SE

if (
    len(df) < 3
    or not np.isfinite([b_x, se_x, b_y, se_y]).all()
    or np.any(b_x == 0)
    or np.any(se_x <= 0)
    or np.any(se_y <= 0)
):
    raise ValueError("请检查 SNP 数量、beta 和 SE 是否有效。")

# 1. 计算每个 SNP 的 Wald 比值和比值 SE
ratios = b_y / b_x
ratio_se = np.sqrt((se_y / b_x)**2 + (b_y * se_x / b_x**2)**2)

# 2. 用等权高斯核密度找到最密集的位置
def simple_mode(values):
    spread = min(
        np.std(values, ddof=1),
        median_abs_deviation(values, scale="normal"),
    )
    bandwidth = max(1e-8, 0.9 * spread / len(values)**0.2)
    grid = np.linspace(
        values.min() - 3 * bandwidth,
        values.max() + 3 * bandwidth,
        512,
    )

    distance = (grid[:, None] - values[None, :]) / bandwidth
    density = np.exp(-0.5 * distance**2).mean(axis=1)
    return grid[np.argmax(density)]

beta_mode = simple_mode(ratios)

# 3. Bootstrap 1,000 次，估计总体效应的 SE
rng = np.random.default_rng(42)
n_boot = 1000
boot_ratios = rng.normal(ratios, ratio_se, size=(n_boot, len(df)))
boot_betas = np.array([simple_mode(draw) for draw in boot_ratios])
se_mode = median_abs_deviation(boot_betas, scale="normal")

if not np.isfinite(se_mode) or se_mode <= 0:
    raise ValueError("Bootstrap 未得到有效的标准误。")

# 4. 输出 OR、95% CI 和双侧 P 值
z = norm.ppf(0.975)
result_simple_mode = pd.DataFrame([{
    "Method": "Simple mode",
    "N_SNP": len(df),
    "Beta": beta_mode,
    "SE": se_mode,
    "OR": np.exp(beta_mode),
    "Lower95": np.exp(beta_mode - z * se_mode),
    "Upper95": np.exp(beta_mode + z * se_mode),
    "P_value": 2 * t.sf(abs(beta_mode / se_mode), df=len(df) - 1),
}])

display(result_simple_mode)

先算 109 个比值。 每个 SNP 都用「对齐后的胆结石 beta ÷ 糖尿病 beta」算一个 Wald 比值

把比值画在一条数轴上，再看哪里最挤。 假设几个 SNP 的比值是 0.019、`0.021`、`0.024`。这三个数虽然各不相同，但都集中在 0.02 附近。代码在每个比值上放一个同样宽、同样高的钟形曲线，再把曲线叠起来。比值越密集，曲线重叠得越多。叠加曲线最高点的横坐标就是简单众数。每个 SNP 在这一步贡献一样。
看这个峰的位置稳不稳。 代码根据每个比值的标准误模拟 1,000 组比值，每组都重新找最高峰，用这些峰位置的波动来估计总体标准误。
这个方法的思路是，如果有效 SNP 的比值聚成了最高的那座峰，一些分散在别处的偏离值就不容易把峰拉走。反过来，如果多个无效 SNP 因为相似的水平多效性聚在一起，最高峰也可能落在有偏的位置。

#加权众数（Weighted mode）

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import median_abs_deviation, norm, t

# 复用简单众数单元里的 ratios、ratio_se 和 boot_ratios。
mode_weights = 1 / ratio_se**2
mode_weights = mode_weights / mode_weights.sum()

def weighted_mode(values, weights):
    spread = min(
        np.std(values, ddof=1),
        median_abs_deviation(values, scale="normal"),
    )
    bandwidth = max(1e-8, 0.9 * spread / len(values)**0.2)
    grid = np.linspace(
        values.min() - 3 * bandwidth,
        values.max() + 3 * bandwidth,
        512,
    )

    distance = (grid[:, None] - values[None, :]) / bandwidth
    kernels = np.exp(-0.5 * distance**2)
    density = np.sum(kernels * weights, axis=1)
    return grid[np.argmax(density)]

# 找原始数据的加权峰，再用 bootstrap 估计标准误。
beta_weighted_mode = weighted_mode(ratios, mode_weights)
boot_estimates = np.array([
    weighted_mode(draw, mode_weights) for draw in boot_ratios
])
se_weighted_mode = median_abs_deviation(boot_estimates, scale="normal")

if not np.isfinite(se_weighted_mode) or se_weighted_mode <= 0:
    raise ValueError("Bootstrap 未得到有效的标准误。")

z = norm.ppf(0.975)
result_weighted_mode = pd.DataFrame([{
    "Method": "Weighted mode",
    "N_SNP": len(ratios),
    "Beta": beta_weighted_mode,
    "SE": se_weighted_mode,
    "OR": np.exp(beta_weighted_mode),
    "Lower95": np.exp(beta_weighted_mode - z * se_weighted_mode),
    "Upper95": np.exp(beta_weighted_mode + z * se_weighted_mode),
    "P_value": 2 * t.sf(
        abs(beta_weighted_mode / se_weighted_mode), df=len(ratios) - 1
    ),
}])

display(pd.concat(
    [result_simple_mode, result_weighted_mode],
    ignore_index=True,
))

#Cochran's Q异质性检测

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2

folder = DATA_DIR
data = pd.read_csv(folder / "T2D_gallstone_harmonized.csv")

b_x = data["b"].to_numpy(dtype=float)                       # 糖尿病 beta
b_y = data["beta_gallstone_aligned"].to_numpy(dtype=float)   # 对齐后的胆结石 beta
se_y = data["se_gallstone"].to_numpy(dtype=float)            # 胆结石 SE

n_snp = len(data)
if (
    n_snp < 2
    or not np.isfinite([b_x, b_y, se_y]).all()
    or np.any(b_x == 0)
    or np.any(se_y <= 0)
):
    raise ValueError("请检查 SNP 数量、beta 和胆结石 SE。")

# 用同一批 SNP 重新计算固定效应 IVW 的斜率
weights = 1 / se_y**2
beta_ivw = np.sum(weights * b_x * b_y) / np.sum(weights * b_x**2)

# Q = 各 SNP 的标准化残差平方之和
expected_b_y = beta_ivw * b_x
Q = np.sum(((b_y - expected_b_y) / se_y)**2)
q_df = n_snp - 1
q_p = chi2.sf(Q, df=q_df)

result_q = pd.DataFrame([{
    "N_SNP": n_snp,
    "IVW_OR": np.exp(beta_ivw),
    "Cochran_Q": Q,
    "df": q_df,
    "Q_P_value": q_p,
}])

display(result_q)

Cochran’s Q 检验用来判断各 SNP 的效应估计是否比抽样误差所能解释的更分散。先假设 109 个 SNP 都在估计同一个真实效应。对每个 SNP，用 IVW 总体斜率乘以它的糖尿病 b_x，得到预测的胆结石 b_y。再用实际估计的 b_y 减去预测值，除以该 SNP 的胆结石标准误 se_y，看它偏离 IVW 直线几个标准误。把这 109 个偏差分别平方后相加，就得到 Q。由于估计了一个总体斜率，自由度为 108。Q 的 P 值表示，在共同效应假设下，观察到当前这么大或更大 Q 的概率。 P 值很小说明各 SNP 的结果比预期更分散，有异质性的证据；P 值较大则表示异质性的证据不足，不能据此证明所有 SNP 都是有效工具。

Q = 180.984，自由度 = 108，P ≈ 0.000014。 这说明 109 个 SNP 给出的效应估计比固定效应 IVW 模型预期的更分散，存在统计学异质性。Q 检验本身无法确定异质性的来源。

#乘性随机效应 IVW

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# 固定效应 IVW 的标准误
se_fixed = 1 / np.sqrt(np.sum(weights * b_x**2))

# 用 Q / 自由度估计额外离散程度，最低设为 1
dispersion = max(1.0, Q / q_df)
se_random = se_fixed * np.sqrt(dispersion)

z = norm.ppf(0.975)
rows = []

for method, se in [
    ("Fixed-effect IVW", se_fixed),
    ("Multiplicative random-effects IVW", se_random),
]:
    rows.append({
        "Method": method,
        "N_SNP": n_snp,
        "Beta": beta_ivw,
        "SE": se,
        "OR": np.exp(beta_ivw),
        "Lower95": np.exp(beta_ivw - z * se),
        "Upper95": np.exp(beta_ivw + z * se),
        "P_value": 2 * norm.sf(abs(beta_ivw / se)),
    })

print(f"Q / df = {Q / q_df:.3f}")
print(f"随机效应 SE 是固定效应的 {np.sqrt(dispersion):.3f} 倍")
display(pd.DataFrame(rows))

乘性随机效应 IVW 是用来处理 不同 SNP 给出的答案比预期更分散 的情况。先用 Cochran’s Q 看这种分歧有多大，再把它算进总体结果的不确定性里。分歧越大，标准误就越大，置信区间也越宽。各 SNP 原来的相对权重一样，所以汇总出来的 OR 保持原值。

在我的分析里，109 个 SNP 给出的结果比预期更分散。把这部分分歧算进去后，OR 仍是 1.040，但 95% 置信区间从 1.003–1.078 扩宽到 0.993–1.090，跨过了代表无效应的 1。也就是说，估计方向还是偏正，但考虑 SNP 之间的分歧后，我们没有那么确定这个效应存在。

# Leave-one-out 逐个去除snp 看剩下的108个的结果

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

folder = DATA_DIR
data = pd.read_csv(folder / "T2D_gallstone_harmonized.csv")

snp_ids = data["SNP"]
b_x = data["b"].to_numpy(dtype=float)
b_y = data["beta_gallstone_aligned"].to_numpy(dtype=float)
se_y = data["se_gallstone"].to_numpy(dtype=float)

if (
    len(data) < 3
    or snp_ids.isna().any()
    or snp_ids.duplicated().any()
    or not np.isfinite([b_x, b_y, se_y]).all()
    or np.any(b_x == 0)
    or np.any(se_y <= 0)
):
    raise ValueError("请检查 SNP 编号、beta 和胆结石 SE。")

def fit_ivw(x, y, se):
    weights = 1 / se**2
    information = np.sum(weights * x**2)
    beta = np.sum(weights * x * y) / information

    q = np.sum(weights * (y - beta * x)**2)
    q_df = len(x) - 1
    se_ivw = np.sqrt(max(1.0, q / q_df) / information)
    margin = norm.ppf(0.975) * se_ivw

    return {
        "N_SNP": len(x),
        "OR": np.exp(beta),
        "Lower95": np.exp(beta - margin),
        "Upper95": np.exp(beta + margin),
        "P_value": 2 * norm.sf(abs(beta / se_ivw)),
    }

# 先用全部 SNP 跑一次，作为比较基准。
full = fit_ivw(b_x, b_y, se_y)

# 每轮只去掉一个 SNP。
rows = []
indices = np.arange(len(data))
for i, snp in enumerate(snp_ids):
    keep = indices != i
    estimate = fit_ivw(b_x[keep], b_y[keep], se_y[keep])
    rows.append({"Removed_SNP": snp, **estimate})

loo = pd.DataFrame(rows)
loo["OR_change"] = loo["OR"] - full["OR"]
loo["Abs_OR_change"] = loo["OR_change"].abs()
loo = loo.sort_values("Abs_OR_change", ascending=False)

output_file = folder / "T2D_gallstone_leave_one_out.csv"
loo.to_csv(output_file, index=False)

n_reversed = ((loo["OR"] - 1) * (full["OR"] - 1) < 0).sum()
print(f"全部 SNP 的 OR：{full['OR']:.4f}")
print(f"逐个剔除后的 OR 范围：{loo['OR'].min():.4f}–{loo['OR'].max():.4f}")
print(f"剔除后效应方向反转的次数：{n_reversed} / {len(loo)}")
print(f"保存位置：{output_file}")

display(loo.head(10))

它会用现有的 109 个 SNP 做 leave-one-out，每次去掉一个 SNP，用剩下的 108 个重跑乘性随机效应 IVW。每轮都会重新计算异质性，让置信区间反映剩余 SNP 的分歧

109 次剔除后的 OR 都大于 1，说明正向点估计对剔除单个位点较稳定。完整模型的置信区间包含 1，而去掉个别位点后，区间下限会在 1 附近变动，所以是否跨过统计显著性阈值对个别位点较敏感

结论：我用FinnGen的胆结石GWAS代替原论文所说的芬兰数据库（找不到），五种方法的点估计都偏正，逐个剔除 SNP 后方向也没有反转。但考虑 SNP 间的异质性后，乘性随机效应 IVW 的 OR 为 1.040（95% CI 0.993–1.090），总体效应仍有不确定性。